In [1]:
import re
import string
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

# Убедитесь, что загрузили необходимые ресурсы NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
df_train = pd.read_csv('./data/train.csv', encoding='1251')
df_test = pd.read_csv('./data/test.csv', encoding='1251')

df_train = df_train[['text', 'sentiment']].dropna()
df_test = df_test[['text', 'sentiment']].dropna()

df_train = df_train[df_train['sentiment'] != 'neutral']
df_test = df_test[df_test['sentiment'] != 'neutral']

print(df_train.shape, df_test.shape)

(16363, 2) (2104, 2)


In [3]:
stop_words = set(stopwords.words('english'))

def encode_target_label(label: str) -> int:
    if label == 'positive':
        return 1
    elif label == 'negative':
        return -1
    else:
        return 0

def preprocess_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r"\\W", " ", text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(rf'[{re.escape(string.punctuation)}]', '', text)
    text = re.sub(r'\n', '', text)
    text = re.sub(r'\w*\d\w*', '', text)
    
    words = word_tokenize(text)
    words = [word for word in words if word not in stop_words]
    words = [word for word in words if len(word) > 1]
    lemmatizer = WordNetLemmatizer()
    words = [lemmatizer.lemmatize(word) for word in words]
    
    return ' '.join(words)

df_train['sentiment'] = df_train['sentiment'].apply(encode_target_label)
df_test['sentiment'] = df_test['sentiment'].apply(encode_target_label)

df_train['text_transformed'] = df_train['text'].apply(preprocess_text)
df_test['text_transformed'] = df_test['text'].apply(preprocess_text)

df_train = df_train[df_train['text_transformed'].str.strip().astype(bool)]
df_test = df_test[df_test['text_transformed'].str.strip().astype(bool)]

### Получение векторов

#### Векторы LSA

In [17]:
full_corpus = pd.concat([df_train['text_transformed'], df_test['text_transformed']])
vectorizer = TfidfVectorizer(max_df=0.5, min_df=5)
vectorizer.fit(full_corpus)

x_train_tfidf = vectorizer.transform(df_train['text_transformed'])
x_test_tfidf = vectorizer.transform(df_test['text_transformed'])

y_train = df_train['sentiment']
y_test = df_test['sentiment']

n_components = 500
lsa = make_pipeline(TruncatedSVD(n_components=n_components, random_state=42), Normalizer(copy=False))
lsa.fit(x_train_tfidf)

x_train_lsa = lsa.transform(x_train_tfidf)
x_test_lsa = lsa.transform(x_test_tfidf)

print(f'Train shape: {x_train_lsa.shape}\nTest shape: {x_test_lsa.shape}')

Train shape: (16357, 500)
Test shape: (2103, 500)


#### Векторы Word2Vec

In [47]:
df_train['tokens'] = df_train['text_transformed'].apply(lambda x: x.split())
df_test['tokens'] = df_test['text_transformed'].apply(lambda x: x.split())

full_tokens = pd.concat([df_train['tokens'], df_test['tokens']])

word2vec_model = Word2Vec(
    sentences=full_tokens,
    vector_size=500,
    window=5,
    min_count=5,
    workers=4,
    epochs=100,
    seed=42
)

In [48]:
def document_vector_w2v(doc_tokens, model):
    valid_tokens = [word for word in doc_tokens if word in model.wv]
    if not valid_tokens:
        return np.zeros(model.vector_size)
    return np.mean(model.wv[valid_tokens], axis=0)

x_train_w2v = np.array([document_vector_w2v(tokens, word2vec_model) for tokens in df_train['tokens']])
x_test_w2v = np.array([document_vector_w2v(tokens, word2vec_model) for tokens in df_test['tokens']])

#### Векторы Doc2Vec

In [50]:
train_tagged = [TaggedDocument(words=doc, tags=[f'TRAIN_{i}']) for i, doc in enumerate(df_train['tokens'])]
test_tagged = [TaggedDocument(words=doc, tags=[f'TEST_{i}']) for i, doc in enumerate(df_test['tokens'])]

all_tagged = train_tagged + test_tagged

doc2vec_model = Doc2Vec(
    documents=all_tagged,
    vector_size=500,
    window=5,
    min_count=2,
    workers=4,
    epochs=40,
    seed=42
)

x_train_d2v = np.array([doc2vec_model.dv[f'TRAIN_{i}'] for i in range(len(df_train))])
x_test_d2v = np.array([doc2vec_model.dv[f'TEST_{i}'] for i in range(len(df_test))])

### Обучение и оценка классификаторов

#### LSA

In [58]:
classifier_lsa = LogisticRegression(random_state=42, max_iter=1000)
classifier_lsa.fit(x_train_lsa, y_train)

train_pred_lsa = classifier_lsa.predict(x_train_lsa)
test_pred_lsa = classifier_lsa.predict(x_test_lsa)

print("LSA")
print(f'Train accuracy: {accuracy_score(y_train, train_pred_lsa):.4f}')
print(classification_report(y_train, train_pred_lsa))

print(f'Test accuracy: {accuracy_score(y_test, test_pred_lsa):.4f}')
print(classification_report(y_test, test_pred_lsa))

LSA
Train accuracy: 0.8691
              precision    recall  f1-score   support

          -1       0.85      0.88      0.86      7777
           1       0.88      0.86      0.87      8580

    accuracy                           0.87     16357
   macro avg       0.87      0.87      0.87     16357
weighted avg       0.87      0.87      0.87     16357

Test accuracy: 0.8645
              precision    recall  f1-score   support

          -1       0.84      0.88      0.86      1000
           1       0.88      0.85      0.87      1103

    accuracy                           0.86      2103
   macro avg       0.86      0.87      0.86      2103
weighted avg       0.87      0.86      0.86      2103



#### Word2Vec

In [59]:
classifier_w2v = LogisticRegression(random_state=42, max_iter=1000)
classifier_w2v.fit(x_train_w2v, y_train)

train_pred_w2v = classifier_w2v.predict(x_train_w2v)
test_pred_w2v = classifier_w2v.predict(x_test_w2v)

print("Word2Vec")
print(f'Train accuracy: {accuracy_score(y_train, train_pred_w2v):.4f}')
print(classification_report(y_train, train_pred_w2v))

print(f'Test accuracy: {accuracy_score(y_test, test_pred_w2v):.4f}')
print(classification_report(y_test, test_pred_w2v))

Word2Vec
Train accuracy: 0.8582
              precision    recall  f1-score   support

          -1       0.85      0.85      0.85      7777
           1       0.87      0.86      0.86      8580

    accuracy                           0.86     16357
   macro avg       0.86      0.86      0.86     16357
weighted avg       0.86      0.86      0.86     16357

Test accuracy: 0.8569
              precision    recall  f1-score   support

          -1       0.84      0.86      0.85      1000
           1       0.87      0.85      0.86      1103

    accuracy                           0.86      2103
   macro avg       0.86      0.86      0.86      2103
weighted avg       0.86      0.86      0.86      2103



#### Doc2Vec

In [60]:
classifier_d2v = LogisticRegression(random_state=42, max_iter=1000)
classifier_d2v.fit(x_train_d2v, y_train)

train_pred_d2v = classifier_d2v.predict(x_train_d2v)
test_pred_d2v = classifier_d2v.predict(x_test_d2v)

print("Doc2Vec")
print(f'Train accuracy: {accuracy_score(y_train, train_pred_d2v):.4f}')
print(classification_report(y_train, train_pred_d2v))

print(f'Test accuracy: {accuracy_score(y_test, test_pred_d2v):.4f}')
print(classification_report(y_test, test_pred_d2v))

Doc2Vec
Train accuracy: 0.8192
              precision    recall  f1-score   support

          -1       0.82      0.80      0.81      7777
           1       0.82      0.84      0.83      8580

    accuracy                           0.82     16357
   macro avg       0.82      0.82      0.82     16357
weighted avg       0.82      0.82      0.82     16357

Test accuracy: 0.8298
              precision    recall  f1-score   support

          -1       0.82      0.82      0.82      1000
           1       0.84      0.84      0.84      1103

    accuracy                           0.83      2103
   macro avg       0.83      0.83      0.83      2103
weighted avg       0.83      0.83      0.83      2103



### Анализ схожести документов

In [22]:
def get_most_similar(doc_vector, doc_vectors, top_n=5):
    similarities = cosine_similarity([doc_vector], doc_vectors)[0]
    similar_indices = similarities.argsort()[-top_n-1:-1][::-1]
    similar_scores = similarities[similar_indices]
    return list(zip(similar_indices, similar_scores))

#### LSA

In [55]:
test_doc_index = 2
test_doc_vector_lsa = x_test_lsa[test_doc_index]

similar_docs_lsa = get_most_similar(test_doc_vector_lsa, x_train_lsa, top_n=5)

print("LSA: Топ 5 схожих документов\n")
print(f"Оригинальный документ: {df_test.iloc[test_doc_index]['text']}")
print(f"Тональность: {df_test.iloc[test_doc_index]['sentiment']}")
print('~' * 50)
for idx, score in similar_docs_lsa:
    print(f"Документ {idx} со схожестью {score:.4f}")
    print(f"Текст: {df_train.iloc[idx]['text']}")
    print(f'Тональность: {y_train.iloc[idx]}')
    print('-' * 50)

LSA: Топ 5 схожих документов

Оригинальный документ:  happy bday!
Тональность: 1
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Документ 1043 со схожестью 1.0000
Текст: HAPPY B-DAY SHARON
Тональность: 1
--------------------------------------------------
Документ 14325 со схожестью 0.8381
Текст: Yaaaaaaay, bday!
Тональность: 1
--------------------------------------------------
Документ 11471 со схожестью 0.7081
Текст:  Ta very much!  Happy B-Day to G-son
Тональность: 1
--------------------------------------------------
Документ 300 со схожестью 0.6420
Текст:  but my bday is JUNE 19.. this is wack... and ihavent seen any promotions for my bday party  someone better finagle this asap!
Тональность: -1
--------------------------------------------------
Документ 10094 со схожестью 0.6398
Текст: TODAY WAS SOO FUN!!  happy bday chrissy <3
Тональность: 1
--------------------------------------------------


#### Word2Vec

In [56]:
test_doc_index = 2
test_doc_vector_w2v = x_test_w2v[test_doc_index]

similar_docs_w2v = get_most_similar(test_doc_vector_w2v, x_train_w2v, top_n=5)

print("Word2Vec: Топ 5 схожих документов\n")
print(f"Оригинальный документ: {df_test.iloc[test_doc_index]['text']}")
print(f"Тональность: {df_test.iloc[test_doc_index]['sentiment']}")
print('~' * 50)
for idx, score in similar_docs_w2v:
    print(f"Документ {idx} со схожестью {score:.4f}")
    print(f"Текст: {df_train.iloc[idx]['text']}")
    print(f'Тональность: {y_train.iloc[idx]}')
    print('-' * 50)

Word2Vec: Топ 5 схожих документов

Оригинальный документ:  happy bday!
Тональность: 1
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Документ 16147 со схожестью 1.0000
Текст:  HaPPy B-DAY Ma Freaaaaaaak  <3
Тональность: 1
--------------------------------------------------
Документ 14325 со схожестью 0.7883
Текст: Yaaaaaaay, bday!
Тональность: 1
--------------------------------------------------
Документ 13658 со схожестью 0.7360
Текст: Had so much fun with jГЇВїВЅ and family  Happy bday my beautiful aunt! s2
Тональность: 1
--------------------------------------------------
Документ 10092 со схожестью 0.7275
Текст: HAPPY JUDDDAY
Тональность: 1
--------------------------------------------------
Документ 10094 со схожестью 0.7189
Текст: TODAY WAS SOO FUN!!  happy bday chrissy <3
Тональность: 1
--------------------------------------------------


#### Doc2Vec

In [57]:
test_doc_index = 2
test_doc_vector_d2v = x_test_d2v[test_doc_index]

similar_docs_d2v = get_most_similar(test_doc_vector_d2v, x_train_d2v, top_n=5)

print("Doc2Vec: Топ 5 схожих документов\n")
print(f"Оригинальный документ: {df_test.iloc[test_doc_index]['text']}")
print(f"Тональность: {df_test.iloc[test_doc_index]['sentiment']}")
print('~' * 50)
for idx, score in similar_docs_d2v:
    print(f"Документ {idx} со схожестью {score:.4f}")
    print(f"Текст: {df_train.iloc[idx]['text']}")
    print(f'Тональность: {y_train.iloc[idx]}')
    print('-' * 50)

Doc2Vec: Топ 5 схожих документов

Оригинальный документ:  happy bday!
Тональность: 1
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Документ 14325 со схожестью 0.9743
Текст: Yaaaaaaay, bday!
Тональность: 1
--------------------------------------------------
Документ 16147 со схожестью 0.9728
Текст:  HaPPy B-DAY Ma Freaaaaaaak  <3
Тональность: 1
--------------------------------------------------
Документ 7307 со схожестью 0.9694
Текст: Happy mother`s day !!!!
Тональность: 1
--------------------------------------------------
Документ 8542 со схожестью 0.9680
Текст: Happy Mother`s Day  http://bit.ly/LRSnG
Тональность: 1
--------------------------------------------------
Документ 4066 со схожестью 0.9640
Текст: In pillow heaven
Тональность: 1
--------------------------------------------------
